## IR Project 2026 | Level 3 | IS

Run this line in the first cell of the notebook so everyone installs the exact same stable versions (compatible with Python 3.13 and Java 8 (jre-1.8) on Windows):

In [ ]:
% pip install -q --force-reinstall ipykernel pandas pyspark==3.5.3 py4j==0.10.9.7

In [ ]:
import math
import os
import re
import pandas as pd
from pyspark.sql import SparkSession

### Person 1 | Ahmed Ali Mohsen

`Positional Index`

* **Positional Index**: فهرس يسجّل لكل كلمة المستندات اللي ظهرت فيها وأماكن ترتيبها (تبدأ من `1` داخل كل مستند).
* **Tokenization**: تحويل النص لحروف صغيرة واستخراج الكلمات بالترتيب بدون حذف Stop Words.

$$\text{Positional Index}(t) = \{ d : [p_1, p_2, \dots, p_k] \}$$

In [ ]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# 1. تشغيل SparkSession بإعدادات مستقرة على ويندوز
spark = (
    SparkSession.builder
    .appName("Person1_Positional_Index")
    .master("local[1]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# 2. مسارات Dataset و Outputs
dataset_dir = os.path.abspath("../Datasets")
output_dir = os.path.abspath("../Outputs")
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "positional_index.txt")

print("Current working dir:", os.getcwd())
print("Looking for Dataset in:", dataset_dir)
print("Dataset exists?:", os.path.exists(dataset_dir))
if os.path.exists(dataset_dir):
    print("Files inside Dataset:", os.listdir(dataset_dir))

# 3. قراءة الملفات (d1.txt إلى d10.txt) وبناء Positional Index
pos_index = {}

for i in range(1, 11):
    doc_id = f"d{i}"
    file_path = os.path.join(dataset_dir, f"{doc_id}.txt")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    rdd = spark.sparkContext.textFile(file_path)
    text = " ".join(rdd.collect()).lower()

    # استخراج الكلمات والترقيم يبدأ من 1 داخل كل ملف بدون حذف Stop Words
    tokens = re.findall(r"\b\w+\b", text)

    for pos, term in enumerate(tokens, start=1):
        pos_index.setdefault(term, {}).setdefault(doc_id, []).append(pos)

# 4. تكوين الجدول
rows = []
for term in sorted(pos_index.keys()):
    for doc_id in sorted(pos_index[term].keys(), key=lambda x: int(x[1:])):
        positions = pos_index[term][doc_id]
        rows.append({
            "Term": term,
            "Document": doc_id,
            "Positions": ", ".join(map(str, positions))
        })

df_pos_index = pd.DataFrame(rows)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
display(df_pos_index)

# 5. حفظ النتيجة في ملف الإخراج
with open(output_file, "w", encoding="utf-8") as f:
    f.write(df_pos_index.to_string(index=False) + "\n")

print(f"\nSaved to: {output_file}")

# 6. إيقاف SparkSession
spark.stop()

### Person 2 | Mazen Emad AbdelSattar

`TF, DF & IDF`

* **TF (Term Frequency)**: عدد مرات ظهور الكلمة جوه المستند الواحد.
* **DF (Document Frequency)**: عدد المستندات اللي ظهرت فيها الكلمة من إجمالي عدد المستندات ($N$).
* **IDF (Inverse Document Frequency)**: وزن يقلل قيمة الكلمات المنتشرة في مستندات كتير ويزود قيمة الكلمات النادرة.

$$\text{IDF}(t) = \log_{10}\left(\frac{N}{\text{DF}(t)}\right)$$

In [ ]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# 1. تشغيل SparkSession بإعدادات مستقرة على ويندوز
spark = (
    SparkSession.builder
    .appName("Person2_TF_DF_IDF")
    .master("local[1]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# 2. مسارات Datasets و Outputs
dataset_dir = os.path.abspath("../Datasets")
if not os.path.exists(dataset_dir):
    dataset_dir = os.path.abspath("../Dataset")

output_dir = os.path.abspath("../Outputs")
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "tf_df_idf.txt")

# 3. قراءة الملفات (d1.txt إلى d10.txt) باستخدام Spark RDD
num_docs = 10
doc_term_counts = {}  # {(term, doc_id): tf}
df_counts = {}        # {term: df}

for i in range(1, num_docs + 1):
    doc_id = f"d{i}"
    file_path = os.path.join(dataset_dir, f"{doc_id}.txt")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    rdd = spark.sparkContext.textFile(file_path)
    text = " ".join(rdd.collect()).lower()
    tokens = re.findall(r"\b\w+\b", text)

    # حساب TF لكل term داخل المستند الحالي
    local_tf = {}
    for term in tokens:
        local_tf[term] = local_tf.get(term, 0) + 1

    # تخزين TF وتحديث DF للمستندات التي ظهر فيها المصطلح
    for term, tf_val in local_tf.items():
        doc_term_counts[(term, doc_id)] = tf_val
        df_counts[term] = df_counts.get(term, 0) + 1

# 4. حساب IDF وتكوين الجدول
rows = []
for term in sorted(df_counts.keys()):
    df_val = df_counts[term]
    idf_val = math.log10(num_docs / df_val)

    # جلب كل المستندات التي ظهر فيها الـ term مرتبة (d1, d2, ..., d10)
    docs_for_term = sorted(
        [d for (t, d) in doc_term_counts.keys() if t == term],
        key=lambda x: int(x[1:])
    )

    for doc_id in docs_for_term:
        tf_val = doc_term_counts[(term, doc_id)]
        rows.append({
            "Term": term,
            "Document": doc_id,
            "TF": tf_val,
            "DF": df_val,
            "IDF": round(idf_val, 4)
        })

df_tf_df_idf = pd.DataFrame(rows)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
display(df_tf_df_idf)

# 5. حفظ النتيجة في ملف الإخراج
with open(output_file, "w", encoding="utf-8") as f:
    f.write(df_tf_df_idf.to_string(index=False) + "\n")

print(f"\nSaved to: {output_file}")

# 6. إيقاف SparkSession
spark.stop()

### Person 3 | Mazen Emad AbdelSattar

`TF-IDF & Normalization`

* **TF-IDF**: بيضرب تكرار الكلمة في المستند (`TF`) في وزن ندرتها (`IDF`) عشان يحدد أهميتها الفعلية في المستند.

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

* **Normalization (L2)**: بيقسم قيمة `TF-IDF` للكلمة على طول متجه المستند كله عشان يوحّد مقياس المستندات باختلاف أطوالها.

$$\text{Norm}(t, d) = \frac{\text{TF-IDF}(t, d)}{\sqrt{\sum_{k \in d} \text{TF-IDF}(k, d)^2}}$$

In [ ]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# 1. تشغيل SparkSession بإعدادات مستقرة على ويندوز
spark = (
    SparkSession.builder
    .appName("Person3_TFIDF_Normalization")
    .master("local[1]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# 2. مسارات Datasets و Outputs
dataset_dir = os.path.abspath("../Datasets")
if not os.path.exists(dataset_dir):
    dataset_dir = os.path.abspath("../Dataset")

output_dir = os.path.abspath("../Outputs")
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "tfidf_normalized.txt")

# 3. قراءة الملفات (d1.txt إلى d10.txt) عبر Spark RDD
num_docs = 10
doc_term_counts = {}  # {(term, doc_id): tf}
df_counts = {}        # {term: df}

for i in range(1, num_docs + 1):
    doc_id = f"d{i}"
    file_path = os.path.join(dataset_dir, f"{doc_id}.txt")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    rdd = spark.sparkContext.textFile(file_path)
    text = " ".join(rdd.collect()).lower()
    tokens = re.findall(r"\b\w+\b", text)

    local_tf = {}
    for term in tokens:
        local_tf[term] = local_tf.get(term, 0) + 1

    for term, tf_val in local_tf.items():
        doc_term_counts[(term, doc_id)] = tf_val
        df_counts[term] = df_counts.get(term, 0) + 1

# 4. حساب IDF و TF-IDF
idf_map = {term: math.log10(num_docs / df_val) for term, df_val in df_counts.items()}

tfidf_records = []
doc_sq_sums = {}  # {doc_id: sum of squared TF-IDF}

for (term, doc_id), tf_val in doc_term_counts.items():
    idf_val = idf_map[term]
    tfidf_val = tf_val * idf_val
    doc_sq_sums[doc_id] = doc_sq_sums.get(doc_id, 0.0) + (tfidf_val ** 2)
    tfidf_records.append({
        "Term": term,
        "Document": doc_id,
        "TF": tf_val,
        "IDF": idf_val,
        "TF_IDF": tfidf_val
    })

doc_norms = {doc_id: math.sqrt(sq_sum) for doc_id, sq_sum in doc_sq_sums.items()}

# 5. حساب L2 Normalized TF-IDF وتكوين الجدول مرتباً
rows = []
for rec in tfidf_records:
    doc_id = rec["Document"]
    norm_val = rec["TF_IDF"] / doc_norms[doc_id] if doc_norms[doc_id] > 0 else 0.0
    rows.append({
        "Term": rec["Term"],
        "Document": doc_id,
        "TF": rec["TF"],
        "IDF": round(rec["IDF"], 4),
        "TF_IDF": round(rec["TF_IDF"], 4),
        "Normalized": round(norm_val, 4)
    })

rows.sort(key=lambda r: (r["Term"], int(r["Document"][1:])))
df_tfidf_norm = pd.DataFrame(rows)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
display(df_tfidf_norm)

# 6. حفظ النتيجة في ملف الإخراج
with open(output_file, "w", encoding="utf-8") as f:
    f.write(df_tfidf_norm.to_string(index=False) + "\n")

print(f"\nSaved to: {output_file}")

# 7. إيقاف SparkSession
spark.stop()

### Person 4 | Youssef Ahmed

`Phrase Queries`

* **Phrase Query**: البحث عن تسلسل كلمات متتالية بنفس الترتيب داخل المستند.
* نستخدم **Positional Index** الناتج من Person 1 للتحقق من أن الكلمة الثانية موجودة في الموقع `(position of first word + 1)`.
* إذا ظهرت `online` في الموقع `5` و `learning` في الموقع `6` → العبارة موجودة.
* إذا ظهرت `online` في الموقع `5` و `learning` في الموقع `8` → العبارة غير موجودة.

$$\text{Match}(q, d) = \exists\, p : p \in \text{pos}(w_1, d) \land (p+1) \in \text{pos}(w_2, d) \land \dots \land (p+k-1) \in \text{pos}(w_k, d)$$

In [ ]:
import os
import re

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# 1. تشغيل SparkSession بنفس إعدادات الشخص الأول (لضمان نفس الـ tokenization)
spark = (
    SparkSession.builder
    .appName("Person4_Phrase_Queries")
    .master("local[1]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# 2. المسارات
dataset_dir = os.path.abspath("../Datasets")
if not os.path.exists(dataset_dir):
    dataset_dir = os.path.abspath("../Dataset")
output_dir = os.path.abspath("../Outputs")
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "query_results.txt")

# 3. إعادة بناء Positional Index (نفس طريقة Person 1 بالظبط)
pos_index = {}  # {term: {doc_id: [positions]}}

for i in range(1, 11):
    doc_id = f"d{i}"
    file_path = os.path.join(dataset_dir, f"{doc_id}.txt")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    rdd = spark.sparkContext.textFile(file_path)
    text = " ".join(rdd.collect()).lower()
    tokens = re.findall(r"\b\w+\b", text)

    for pos, term in enumerate(tokens, start=1):
        pos_index.setdefault(term, {}).setdefault(doc_id, []).append(pos)

# 4. دالة البحث عن عبارة داخل الـ Positional Index
def phrase_query(phrase, positional_index):
    """ترجع قائمة doc_ids اللي بتحتوي على العبارة كتسلسل متتالي بنفس الترتيب."""
    tokens = re.findall(r"\b\w+\b", phrase.lower())
    if not tokens:
        return []
    if len(tokens) == 1:
        return sorted(positional_index.get(tokens[0], {}).keys(), key=lambda d: int(d[1:]))

    first_term = tokens[0]
    candidate_docs = set(positional_index.get(first_term, {}).keys())
    matching_docs = []

    for doc_id in sorted(candidate_docs, key=lambda d: int(d[1:])):
        first_positions = positional_index[first_term][doc_id]
        phrase_found = False

        for start_pos in first_positions:
            ok = True
            for offset, next_term in enumerate(tokens[1:], start=1):
                expected_pos = start_pos + offset
                next_positions = positional_index.get(next_term, {}).get(doc_id, [])
                if expected_pos not in next_positions:
                    ok = False
                    break
            if ok:
                phrase_found = True
                break  # لقينا العبارة في المستند ده

        if phrase_found:
            matching_docs.append(doc_id)

    return matching_docs

# 5. تنفيذ الاستعلامات المطلوبة
queries = ["online learning", "student assessment"]
results = {q: phrase_query(q, pos_index) for q in queries}

# متغيرات جاهزة لتسليمها لـ Person 5 (Boolean Queries)
online_learning_docs   = results["online learning"]
student_assessment_docs = results["student assessment"]

# 6. عرض النتائج
print("=" * 50)
print("Phrase Query Results")
print("=" * 50)
for q, docs in results.items():
    print(f"Query: '{q}'")
    print(f"Matching Documents: {docs}")
    print(f"Number of matches : {len(docs)}")
    print("-" * 50)

# 7. حفظ النتائج في Outputs/query_results.txt
with open(output_file, "w", encoding="utf-8") as f:
    f.write("Phrase Query Results\n")
    f.write("=" * 50 + "\n")
    for q, docs in results.items():
        f.write(f"Query: '{q}'\n")
        f.write(f"Matching Documents: {docs}\n")
        f.write(f"Number of matches : {len(docs)}\n")
        f.write("-" * 50 + "\n")

print(f"\nSaved to: {output_file}")

# 8. إيقاف SparkSession
spark.stop()

### Person 5 | 

### Person 6 | 